# Parametric trigger mimic (tabular)

Features resemble weather/AQI/traffic inputs; labels use the same threshold logic as `TRIGGERS` in Oasis (heat+sustained hours, rain rate, AQI band, TomTom-style congestion ratio + confidence). The model learns a smooth boundary; production still uses explicit rules for audit—this is for calibration experiments and shadow scoring.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "scripts"))

from synthetic_data import generate_trigger_tabular_rows
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

df = generate_trigger_tabular_rows(12000)
df.head()

In [ ]:
features = ["temp_c", "heat_hours", "rain_mm_h", "aqi", "traffic_speed_ratio", "traffic_confidence"]
X, y = df[features], df["trigger_any"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GradientBoostingClassifier(random_state=42, max_depth=4, n_estimators=120, learning_rate=0.08)),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print("holdout accuracy", accuracy_score(y_test, pred))
print("(Generator uses noisy observations vs latent truth for labels — expect < 1.0)")
print(classification_report(y_test, pred))

In [ ]:
import joblib
ROOT.joinpath("artifacts").mkdir(parents=True, exist_ok=True)
ROOT.joinpath("data/synthetic").mkdir(parents=True, exist_ok=True)
joblib.dump(pipe, ROOT / "artifacts" / "trigger_tabular.joblib")
df.to_csv(ROOT / "data" / "synthetic" / "trigger_tabular.csv", index=False)